# Warpsmith on a cloud GPU

Works on Google Colab (Runtime > Change runtime type > T4 GPU) and Kaggle (Settings > Accelerator > GPU, Internet on).

Milestone 1: run the evaluation harness on hand-written matmul kernels. No LLM yet.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import os

REPO_URL = "https://github.com/SupreetTadeparti/warpsmith.git"
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORKDIR)
if os.path.exists("warpsmith"):
    !git -C warpsmith pull --ff-only
else:
    !git clone {REPO_URL}
os.chdir(f"{WORKDIR}/warpsmith")
!pip install -q -e ".[dev]"

In [ ]:
!python -m warpsmith info

In [ ]:
# GPU tests run here (they skip on machines without nvcc), including the red-team suite.
!python -m pytest -q -rs

In [ ]:
!python -m warpsmith eval matmul problems/matmul/naive.cu --no-compare

In [ ]:
# Tiled kernel, compared against the naive kernel and cuBLAS (via torch.matmul).
!mkdir -p runs
!python -m warpsmith eval matmul examples/matmul/tiled.cu --json runs/tiled.json

In [ ]:
# Watch the harness catch individual cheats.
!python -m warpsmith eval matmul tests/redteam/matmul/pointer_cache.cu --no-compare
!python -m warpsmith eval matmul tests/redteam/matmul/legacy_stream.cu --no-compare
!python -m warpsmith eval matmul tests/redteam/matmul/oob_far.cu --sanitize --no-bench

## The agent (gpt-oss-120b via UF NaviGator)

One-time setup: add your NaviGator key as a secret named `NAVIGATOR_API_KEY`.
- Colab: key icon in the left sidebar > Add new secret > turn on Notebook access.
- Kaggle: Add-ons > Secrets.

Never paste the key into a cell.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["NAVIGATOR_API_KEY"] = userdata.get("NAVIGATOR_API_KEY")
except ImportError:
    from kaggle_secrets import UserSecretsClient
    os.environ["NAVIGATOR_API_KEY"] = UserSecretsClient().get_secret("NAVIGATOR_API_KEY")
os.environ["NAVIGATOR_BASE_URL"] = "https://api.navigator.ai.ufl.edu"
os.environ["WARPSMITH_MODEL"] = "gpt-oss-120b"
print("key loaded:", bool(os.environ["NAVIGATOR_API_KEY"]))

In [ ]:
# One tiny request to confirm the endpoint, model name and key work.
!python -m warpsmith llm-check

In [ ]:
# Each iteration: the model proposes a kernel, the harness checks and times it, feedback goes back.
# Roughly 1-3 minutes per iteration. Every prompt, response and kernel is saved under runs/.
!python -m warpsmith agent matmul --iterations 10

In [ ]:
import glob
latest = sorted(glob.glob('runs/matmul-*'))[-1]
print(latest)
print(open(f'{latest}/summary.json').read())
print(open(f'{latest}/best.cu').read())

In [ ]:
# Colab: download the whole run (prompts, responses, every kernel) as a zip.
!cd runs && zip -qr ../runs.zip .
try:
    from google.colab import files
    files.download('runs.zip')
except ImportError:
    print('On Kaggle: runs.zip is in the Output panel')